# Lexicon maintenance - non-English gloss backfill (phase 8)

Thin driver for the backfill loop; every operation is a package function in
`lang_tools.lexicon.ingestion.gloss_backfill` and `lang_tools.llm.runner`.

1. **Worklist** - `missing_glosses`: every (concept, language) with a member word
   but no gloss, grounded in the English gloss and both languages' members.
2. **Sample** - a live run of a few chunks; read the measured usage and cost.
3. **Full run** - one OpenAI batch (`submit_batch`, tagged so it can be found
   again), collected with `collect_batch` from any checkout within 30 days.
4. **Check** - `check_glosses` screens every gloss into `proposed` / `rejected`
   rows of the committed `data/curated/gloss_backfill.jsonl`.
5. **Review** - read `review_sample`; `accept_run` flips the languages that pass.
6. **Rebuild** - `02_transform` applies the accepted rows (gaps only, tagged `llm`,
   normalized by `normalize_gloss`), then re-run the quality gate.

The first run (2026-10-21) used `gpt-6-luna` + prompt v3, reasoning off; its model
comparison and review live under `scratch_space/09_concept_model/08_maintenance/`.

**Paid-run rule.** Every LLM job here follows the same order: run a small live sample,
read its measured usage and the extrapolated cost, compare models if the choice is
open, and only then flip `RUN_PAID` and submit the full batch. Nothing in this notebook
spends money until that flag is set by hand.

In [ ]:
import collections

from llm_core.chat.config.openai import ChatOpenAIConfig

from lang_tools.lexicon.ingestion.gloss_backfill import accept_run
from lang_tools.lexicon.ingestion.gloss_backfill import check_glosses
from lang_tools.lexicon.ingestion.gloss_backfill import chunk_items
from lang_tools.lexicon.ingestion.gloss_backfill import missing_glosses
from lang_tools.lexicon.ingestion.gloss_backfill import review_sample
from lang_tools.lexicon.maintenance import GLOSS_BACKFILL_FILE
from lang_tools.lexicon.maintenance import GlossProposal
from lang_tools.lexicon.maintenance import curated_path
from lang_tools.lexicon.maintenance import read_jsonl
from lang_tools.lexicon.maintenance import write_jsonl
from lang_tools.llm.gloss_translate import build_gloss_translate_chain
from lang_tools.llm.runner import collect_batch
from lang_tools.llm.runner import load_checkpoint
from lang_tools.llm.runner import read_records
from lang_tools.llm.runner import run_chain
from lang_tools.llm.runner import submit_batch
from lang_tools.llm.runner import summarize_usage
from lang_tools.params.lang_tools_params import get_lang_tools_params
from lang_tools.params.load_env import load_env

load_env()
data_fol = get_lang_tools_params().paths.data_fol
work = data_fol / "_raw" / "lexicon" / "maintenance" / "gloss_backfill"
curated = curated_path(data_fol, GLOSS_BACKFILL_FILE)

MODEL = "gpt-6-luna"
PROMPT_VERSION = "3"
JOB_TAG = {"job": "lang-tools-gloss-backfill", "prompt": f"gloss_translate/v{PROMPT_VERSION}"}
#: Batch prices per 1M tokens for the chosen model (check the OpenAI pricing page
#: before a run; the standard live tier is double).
PRICES = {"input_per_m": 0.05, "cached_per_m": 0.005, "output_per_m": 0.25}
RUN_PAID = False  # flip by hand only after reading the sample's measured cost


class NoReasoningConfig(ChatOpenAIConfig):
    """ChatOpenAIConfig with reasoning off (llm-core has no kwargs passthrough)."""

    reasoning_effort: str = "none"


chain = build_gloss_translate_chain(NoReasoningConfig(model=MODEL), version=PROMPT_VERSION)

## Worklist

In [ ]:
items = missing_glosses(data_fol)
chunks = chunk_items(items)
by_key = {item.key: item for item in items}
len(items), len(chunks), collections.Counter(item.language for item in items)

## Sample (live, a few chunks)

Measured usage on the sample, scaled to the whole worklist. The live tier costs
about twice the batch prices above.

In [ ]:
sample_cp = work / "sample.jsonl"
run_chain(chain, chunks, checkpoint=sample_cp, limit=4, concurrency=4)
totals = summarize_usage(read_records(sample_cp))
sampled = sum(len(inp.concepts) for key, inp in chunks if key in load_checkpoint(sample_cp, chain.output_model))
scale = len(items) / max(sampled, 1)
totals, f"estimated full run at batch prices: ${totals.cost(**PRICES) * scale:.2f}"

## Full run (one batch)

Re-running this cell never pays twice: while the batch is alive or completed,
`submit_batch` returns its id. Commit the printed id (or rely on `JOB_TAG`:
`runner.find_batches(JOB_TAG)` lists it). Then run the collect cell until it
reports `completed` (a batch has 24 h; its output is deleted 30 days after).

In [ ]:
batch_cp = work / "run.jsonl"
if RUN_PAID:
    batch_id = submit_batch(
        chain,
        chunks,
        checkpoint=batch_cp,
        workdir=work,
        extra_body={"reasoning_effort": "none"},
        metadata=JOB_TAG,
    )
    print(batch_id)

In [ ]:
# collect_batch(chain, batch_id, checkpoint=batch_cp)

## Check and write the curated file

In [ ]:
outputs = load_checkpoint(batch_cp, chain.output_model)
proposals = []
for output in outputs.values():
    found = [by_key[g.key] for g in output.glosses if g.key in by_key]
    proposals += check_glosses(found, output)
write_jsonl(proposals, curated)
usage = summarize_usage(read_records(batch_cp))
collections.Counter((p.language, p.status) for p in proposals), f"${usage.cost(**PRICES):.2f}"

## Review and accept

Read the sample (and spot-check the rejected rows; the model sometimes echoes a
key with a typo, leaving that gloss missing). Then accept the languages whose
sample passes.

In [ ]:
proposals = read_jsonl(GlossProposal, curated)
[(p.language, by_key[f"{p.language}:{p.concept_id.rsplit('__', 1)[1]}"].english_definition, p.proposed_definition)
 for p in review_sample(proposals, per_language=10)]

In [ ]:
# write_jsonl(accept_run(proposals, languages=["es", "fr", "it", "pt"]), curated)

## Rebuild

Re-run `notebooks/lexicon_ingest/02_transform.ipynb`, then the quality gate
(`scratch_space/09_concept_model/05.4_data_quality/05.4_data_quality_checks.ipynb`).
The build logs `Gloss backfill: N filled`; the gate's gloss-coverage table should
read 100% for the accepted languages.